In [1]:
import os
import re
import math
import json
from dotenv import load_dotenv

from groq import Groq

import sys

from src.tool_pattern.tool import tool
from src.utils.extraction import extract_tag_content

load_dotenv()

MODEL = "llama-3.1-8b-instant"
GROQ_CLIENT = Groq()

In [41]:
REACT_SYSTEM_PROMPT = """
You are a function calling AI model. You operate by running a loop with the following steps: Thought, Action, Observation.
You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don' make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.

For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools / actions:

<tools> 
%s
</tools>

Example session:

<question>What's the current temperature in Madrid?</question>
<thought>I need to get the current weather in Madrid</thought>
<tool_call>{"name": "get_current_weather","arguments": {"location": "Madrid", "unit": "celsius"}, "id": 0}</tool_call>

You will be called again with this:

<observation>{0: {"temperature": 25, "unit": "celsius"}}</observation>

You then output:

<response>The current temperature in Madrid is 25 degrees Celsius</response>

Additional constraints:

- If the user asks you something unrelated to any of the tools above, answer freely enclosing your answer with <response></response> tags.
"""

In [42]:
from src.research_articule.language_learning_news import collect_language_learning_news
from src.research_articule.article_paragraph_collector import collect_article_paragraphs

#collect_language_learning_news(articles_count=3, paragraphs_per_article=3, output_file='french_language_learning_news.txt')
#collect_article_paragraphs(max_items=2, max_paragraphs_per_article=3, output_file='article_paragraphs.txt')

# @tool
# def get_language_learning_news() -> str:
#     """
#     Extract paragraphs from an French article.
#     """
#     return collect_language_learning_news(articles_count=3, paragraphs_per_article=3, output_file='french_language_learning_news.txt')

@tool
def get_language_learning_news(
    articles_count: int = 3,
    paragraphs_per_article: int = 3,
    output_file: str = "french_language_learning_news.txt"
) -> str:
    """
    Collect French language-learning news articles.
    """

    return collect_language_learning_news(
        articles_count=articles_count,
        paragraphs_per_article=paragraphs_per_article,
        output_file=output_file
    )


@tool
def get_article_paragraphs() -> str:
    """
    Extract paragraphs from an English article.
    """
    return collect_article_paragraphs(max_items=3, max_paragraphs_per_article=3, output_file='article_paragraphs.txt')

@tool
def read_txt_file(file_path: str) -> str:
    """
    Reads a text file and returns its content as a string.

    Args:
        file_path (str): Path to the .txt file.

    Returns:
        str: Content of the file.
    """
    file_path = 'french_language_learning_news.txt'
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()

    return content

In [30]:
collect_article_paragraphs(max_items=3, max_paragraphs_per_article=3, output_file='article_paragraphs.txt')

'Collected article paragraphs:\n1. Married at First Sight UK brides tell BBC they were raped by on-screen husbands\n   Link: https://www.bbc.com/news/articles/cp8pz1k4r2lo?at_medium=RSS&at_campaign=rss\n   Paragraph 1: Warning: contains details of alleged sexual offences and misconduct\n   Paragraph 2: Two women have told the BBC they were raped during the filming of one of Channel 4\'s biggest shows, Married at First Sight UK, while a third has described an allegation of a non-consensual sex act.\n   Paragraph 3: The show did not do enough to protect them, they all said.\n2. PM insists he will not walk away as Burnham promises to change Labour\n   Link: https://www.bbc.com/news/articles/cx21en4807wo?at_medium=RSS&at_campaign=rss\n   Paragraph 1: Sir Keir Starmer has repeated his insistence that he will "not walk away" from the job of prime minister despite more than a week of turbulence in his party, which saw five of his ministers resign.\n   Paragraph 2: He said the last 10 days had

In [43]:
available_tools = {
    "get_language_learning_news": get_language_learning_news,
    "get_article_paragraphs": get_article_paragraphs,
    "read_txt_file": read_txt_file
}

In [44]:
print("Tool name: ", get_language_learning_news.name)
print("Tool signature: ", read_txt_file.fn_signature)

Tool name:  get_language_learning_news
Tool signature:  {"name": "read_txt_file", "description": "\n    Reads a text file and returns its content as a string.\n\n    Args:\n        file_path (str): Path to the .txt file.\n\n    Returns:\n        str: Content of the file.\n    ", "parameters": {"properties": {"file_path": {"type": "str"}}}}


In [45]:
tools_signature = get_language_learning_news.fn_signature + ",\n" + get_article_paragraphs.fn_signature + ",\n" + read_txt_file.fn_signature

In [46]:
REACT_SYSTEM_PROMPT = REACT_SYSTEM_PROMPT % tools_signature

In [47]:
print(REACT_SYSTEM_PROMPT)


You are a function calling AI model. You operate by running a loop with the following steps: Thought, Action, Observation.
You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don' make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.

For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools / actions:

<tools> 
{"name": "get_language_learning_news", "description": "\n    Collect French language-learning news articles.\n    ", "parameters": {"properties": {"articles_count": {"type": "int"}, "paragraphs_per_article": {"type": "int"}, "output_file": {"type": "str"}}}},
{"name": "get_artic

In [38]:
from src.REACT.react_agent import ReactAgent

In [48]:
agent = ReactAgent(tools=[get_language_learning_news, get_article_paragraphs, read_txt_file])

In [49]:
agent.run(user_msg="I want some french news")


Thought: I need to get some French news articles to provide to the user.

Using Tool: get_language_learning_news

Tool call dict: 
{'name': 'get_language_learning_news', 'arguments': {'articles_count': 5, 'paragraphs_per_article': 3, 'output_file': 'french_news.txt'}, 'id': 0}

Tool result: 
[{'date': 'lundi 18 mai 2026 à 07:08:13', 'link': 'https://www.bbc.com/afrique/articles/c62x0pe4x6qo', 'name': 'L’OMS déclare que l’épidémie d’Ebola en RD Congo est une urgence internationale', 'info': "L'Organisation mondiale de la santé (OMS) a déclaré qu'une épidémie d'Ebola en République démocratique du Congo constituait une urgence de santé publique de portée internationale.\n\nL'agence a indiqué que l'épidémie dans la province de l'Ituri, dans l'est de la RD Congo, qui a enregistré environ 246 cas suspects et 80 décès signalés, ne répond pas aux critères d'une urgence pandémique.\n\nMais elle a prévenu qu'il pourrait s'agir « d'une épidémie beaucoup plus importante » que ce qui est actuellem

'Voicez les Actualités en français pour vous:\n1. L’OMS déclare que l’épidémie d’Ebola en RD Congo est une urgence internationale.\n2. Découvrez les bienfaits de la consommation de pattes de poulet pour la santé.\n3. Trump et Xi concluent des pourparlers « très fructueux », mais peu d’accords ont été confirmés.\n4. Sur les traces du soldat Tom : élucider le mystère familial d’un prisonnier de guerre soviétique de la Seconde Guerre mondiale.\n5. 3 propriétés qui font des sardines un superaliment bon marché idéal pour votre santé.'

In [50]:
agent.run(user_msg="I want some french news and out of this news, propose me a conversation to practice french vocabulary")


Thought: I need to get some French language-learning news articles and then use them to propose a conversation to practice French vocabulary.

Using Tool: get_language_learning_news

Tool call dict: 
{'name': 'get_language_learning_news', 'arguments': {'articles_count': 5, 'paragraphs_per_article': 3, 'output_file': 'french_news.txt'}, 'id': 0}

Tool result: 
[{'date': 'lundi 18 mai 2026 à 17:28:45', 'link': 'https://www.bbc.com/afrique/articles/c86d5379914o', 'name': 'Qu’est-ce que le virus Ebola et pourquoi est-il si difficile de stopper cette épidémie\xa0?', 'info': "L'Organisation mondiale de la santé (OMS) a déclaré que l'épidémie d'Ebola en République démocratique du Congo constituait une urgence de santé publique de portée internationale.\n\nIl est difficile de faire face à cette épidémie, car elle concerne une souche rare contre laquelle il n'existe aucun vaccin et les cas ont été détectés dans une région en proie à un conflit.\n\nLes virus Ebola infectent normalement les anim

'Let\'s practice our French vocabulary with a conversation. We can start with the topic of the Ebola epidemic. What do you think about the current situation in the Democratic Republic of Congo? \n\nVous pouvez commencer par dire quelque chose comme : "Je suis inquiet pour les personnes touchées par l\'épidémie" ou "Je pense que l\'OMS devrait faire plus pour aider les pays affectés".\n\nOr we could discuss the art and culture scene in cities like London, Paris, and New York. Which city would you like to visit and why?\n\nVous pouvez dire quelque chose comme : "J\'aimerais visiter Londres pour voir les musées et les théâtres" ou "Je préfère Paris pour son atmosphère artistique et ses cafés".\n\nOr if you\'re interested in health and nutrition, we could talk about the benefits of eating chicken feet. Have you ever tried them before?\n\nVous pouvez dire quelque chose comme : "Oui, j\'ai déjà mangé des pattes de poulet et je les ai trouvées délicieuses" ou "Non, je n\'ai jamais essayé, mai

In [51]:
agent.run(user_msg="Je préfère Paris pour son atmosphère artistique et ses cafés")

"Paris est indeed connu pour son atmosphère artistique unique et ses cafés historiques. C'est une ville qui inspire la créativité et offre une expérience culturelle riche."